In [3]:
import pandas as pd
import numpy as np
import ast
import os
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


# 1. Expand the SOC Codes

In [4]:
# Open up the mca
mca_df = pd.read_csv('../data/auxiliary/mca_soc_codes.csv', dtype={'PSOC Code':str, 'ISCO Code':str})
mca_df['Educational Qualification'] = (
    mca_df['Educational Qualification'].str.strip().replace({'No Match Found': ''})
)

# this is for the experiment where we let the SOC codes be expanded
# so we will let a whole family of jobs enter the list per Job title based on if you have the same four codes
mca_df['2019 SOC Codes'] = mca_df['2019 SOC Codes'].apply(ast.literal_eval)

In [5]:
# get the mapping of 2019 SOC codes
soc_codes_2019_df = pd.read_csv(
    '../data/labor_codes/2019_to_SOC_Crosswalk.csv',
    usecols=[0, 1],
)
soc_codes_2019_map = dict(
    zip(soc_codes_2019_df['O*NET-SOC 2019 Code'], 
        soc_codes_2019_df['O*NET-SOC 2019 Title'])
)

In [6]:
mca_df['SOC Codes'] = mca_df['2019 SOC Codes']

# 2. Get the embedding of each PSOC task

In [ ]:
# Open up the json of psoc and soc tasks
with open('../data/auxiliary/psoc_tasks_map.json', 'r') as file:
    psoc_tasks_map = json.load(file)

with open('../data/auxiliary/embedded_soc_tasks_map.json', 'r') as file:
    embedded_soc_tasks_map = json.load(file)

In [ ]:
# Map the PSOC tasks and the embedded SOC tasks
mca_df['PSOC Tasks'] = mca_df['PSOC Code'].map(psoc_tasks_map)

def get_embedded_soc_tasks(codes, embedded_soc_tasks_map):
    """
        Get embedded SOC tasks for available SOC codes.
        If it is not available, fallback and get the tasks
        of all related jobs in the minor group
    """

    embedded_tasks = []

    for code in codes:
        # Exact match
        if code in embedded_soc_tasks_map:
            embedded_tasks.append(embedded_soc_tasks_map[code])
            continue

        # Fallback: same first 4 digits
        prefix = code[:5]

        for soc_code, tasks in embedded_soc_tasks_map.items():
            if soc_code[:5] == prefix:
                embedded_tasks.append(tasks)

    return embedded_tasks

# embed the SOC Codes
mca_df['Embedded SOC Tasks'] = mca_df['SOC Codes'].apply(
    get_embedded_soc_tasks,
    embedded_soc_tasks_map=embedded_soc_tasks_map
)

# create PSOC tasks that have context by adding the job title
mca_df['PSOC Tasks with Context'] = mca_df.apply(
    lambda job: [
        f"{job['Job Title']}, "
        f"{task}"
        for task in job['PSOC Tasks']
    ] if isinstance(job['PSOC Tasks'], list) else job['PSOC Tasks'],
    axis=1
)

# create MCA job titles have context by adding education and sector
mca_df['Titles with Context'] = mca_df.apply(
    lambda job: 
        f"{job['Job Title']}, "
        f"{job['Educational Qualification']}, "
        f"{job['Job Sector']}"
    ,
    axis=1
)

In [ ]:
# the code here is kinda weird because if the file has been made already
# we have to do some fixing of the SOC codes, but if it was newly made,
# then, the SOC Codes field is already a list

if IS_EXPANDED:
    output_path = "../data/auxiliary/expanded_mca_soc_embedded.csv"
else:
    output_path = "../data/auxiliary/mca_soc_embedded.csv"

if os.path.exists(output_path):
    mca_df = pd.read_csv(output_path)
    # Convert the saved string representations back into lists
    mca_df['Embedded PSOC Tasks with Context'] = (
        mca_df['Embedded PSOC Tasks with Context']
        .apply(ast.literal_eval)
    )

    # deal with the annoying fact that CSVs dont save lists
    mca_df['SOC Codes'] = (
        mca_df['SOC Codes']
        .apply(ast.literal_eval)
    )

    mca_df['Embedded SOC Tasks'] = mca_df['SOC Codes'].apply(
        get_embedded_soc_tasks,
        embedded_soc_tasks_map=embedded_soc_tasks_map
    )

else:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # create the embeddings per task for each row
    mca_df['Embedded PSOC Tasks with Context'] = mca_df[
        'PSOC Tasks with Context'
    ].apply(
        lambda tasks_with_context: [
            embedding_model.encode(
                task_with_context,
                normalize_embeddings=True
            ).tolist()
            for task_with_context in tasks_with_context
        ] if isinstance(tasks_with_context, list) else tasks_with_context
    )
    mca_df.to_csv(output_path, index=False)

In [ ]:
# embed also the names of the job and of the SOC codes
soc_df = pd.read_csv(
    '../data/labor_codes/2019_to_SOC_Crosswalk.csv',
    usecols=[0, 1],
    names=['SOC', 'Title'], 
    skiprows=1
)

soc_code_title_map = dict(zip(soc_df.SOC, soc_df.Title))

In [ ]:
# make sure SOC Codes is actually filled with lists and NOT a string
mca_df['SOC Titles'] = mca_df['SOC Codes'].apply(
    lambda codes : [soc_code_title_map[code] for code in codes]
)

# Embed the Job Title and the SOC Codes
embedding_model = SentenceTransformer("all-mpnet-base-v2")
mca_df['Embedded Job Title'] = mca_df['Job Title'].apply(
    lambda job_title : [embedding_model.encode(job_title, normalize_embeddings=True).tolist()]
)

mca_df['Embedded SOC Titles'] = mca_df['SOC Titles'].apply(
    lambda SOC_Titles: [
        embedding_model.encode(
            SOC_Title,
            normalize_embeddings=True
        ).tolist()
        for SOC_Title in SOC_Titles
    ])

# 2. Choose the most representative SOC code for each Occupation based on Cosine Similarity

For each PSOC occupation, we compare its tasks and job title against those of each candidate SOC code using cosine similarity between their embeddings.

For the tasks, we find the most similar SOC task for each PSOC task and take the mean of these best-match scores. Separately, we calculate the cosine similarity between the PSOC job title and the SOC title.

We then take the mean of the task similarity and title similarity to obtain the overall PSOC-SOC similarity score.

**PSOC tasks + job title -> task similarity + title similarity -> average -> representative SOC code**

A higher overall score indicates that the SOC is more semantically representative of the PSOC occupation.

In [ ]:
def estimate_representative_soc_codes(job, verbose=False):
    """
    Rank candidate SOC codes from most to least representative.

    Each candidate SOC code is evaluated using:
    1. Task similarity: mean of the best cosine similarity for each
       PSOC task against the candidate SOC's tasks.
    2. Title similarity: cosine similarity between the PSOC job title
       and the candidate SOC title.

    The final score is the mean of the task and title similarities.

    Returns:
        List of tuples: [(SOC code, score), ...], sorted descending by score.
    """
    # if the SOC Code from 2019 was just one or two, leave it alone
    if (len(job['2019 SOC Codes']) == 1):
        return job['2019 SOC Codes']

    # Convert embedded PSOC tasks and job title
    embedded_psoc_tasks = np.array(
        job['Embedded PSOC Tasks with Context']
    )

    embedded_job_title = np.array(
        job['Embedded Job Title']
    ).reshape(1, -1)

    soc_scores = []

    # Evaluate every candidate SOC code
    for embedded_soc_tasks, embedded_soc_title, soc_code in zip(
        job['Embedded SOC Tasks'],
        job['Embedded SOC Titles'],
        job['SOC Codes']
    ):
        # Task similarity
        soc_tasks_array = np.array(embedded_soc_tasks)

        similarity_matrix = cosine_similarity(
            embedded_psoc_tasks,
            soc_tasks_array
        )

        best_similarity_scores = similarity_matrix.max(axis=1)

        task_similarity = best_similarity_scores.mean()

        # Title similarity
        soc_title_embedding = np.array(
            embedded_soc_title
        ).reshape(1, -1)

        title_similarity = cosine_similarity(
            embedded_job_title,
            soc_title_embedding
        )[0, 0]

        # Final score
        mean_similarity = (task_similarity + title_similarity) / 2
        mean_similarity = task_similarity * 0.6 + title_similarity * 0.4

        soc_scores.append({
            'SOC Code': soc_code,
            'Task Similarity': task_similarity,
            'Title Similarity': title_similarity,
            'Score': mean_similarity
        })

    # Sort from most likely to least likely
    soc_scores = sorted(
        soc_scores,
        key=lambda x: x['Score'],
        reverse=True
    )

    result_soc_scores = [
        soc_score['SOC Code'] for soc_score in soc_scores
    ]

    if verbose:
        for rank, result in enumerate(soc_scores, start=1):
            print(
                f"{rank}. {result['SOC Code']}: "
                f"Task = {result['Task Similarity']:.4f}, "
                f"Title = {result['Title Similarity']:.4f}, "
                f"Average = {result['Score']:.4f}"
            )

    return result_soc_scores

In [ ]:
# get the most representative SOC codes
mca_df['Sorted SOC Codes'] = mca_df.apply(estimate_representative_soc_codes, axis=1)
mca_df['Sorted SOC Titles'] = mca_df['Sorted SOC Codes'].apply(
    lambda soc_codes : [
        soc_code_title_map[soc_code] for soc_code in soc_codes
    ])

mca_df['SOC Code'] = mca_df['Sorted SOC Codes'].apply(
    lambda soc_codes : soc_codes[0]
)
mca_df['SOC Title'] = mca_df['Sorted SOC Titles'].apply(
    lambda soc_titles : soc_titles[0]
)

In [ ]:
relevant_cols = [
    'Job Title',
    'Educational Qualification',
    'Job Sector',
    'Educational Pathway',
    'HEI with PRC Exam',
    'Some HEI', 
    'Job Subsector',
    'PSOC Code',
    'ISCO Code',
    'SOC Code',
    'SOC Title',
    'SOC Codes',
    'SOC Titles'
]
if IS_EXPANDED:
    mca_df[relevant_cols].to_csv('../data/auxiliary/final_expanded_mca_soc_code.csv', index=False)
else:
    mca_df[relevant_cols].to_csv('../data/auxiliary/final_mca_soc_code.csv', index=False)

The two examples below demonstrate how accurately the system identifies the most representative SOC code. 

For example, the teachers in the first table had 37 possible SOC codes to choose from, while the occupational therapist in the second table had 19. However, the previous approach would naively assign the same general SOC code repeatedly across the different teacher occupations, which is as 25-1199.00 (Postsecondary Teachers, All Other). This results in a loss of variety and specificity in the mappings. Similarly, the occupational therapist could be assigned the general code 31-9099.00 (Healthcare Support Workers, All Other). 

With the new system, the combination of task and title similarity allows the system to distinguish between occupations and select more specific and representative SOC codes from the available candidates.

In [ ]:
mca_df.loc[97, 'PSOC Code']

'5411'

In [ ]:
mca_df[['Job Title', 'SOC Title']].sample(50)

,Job Title,SOC Title
474,Hospitality Systems and Technology Analyst,Computer Systems Engineers/Architects
154,Church Worker,"Religious Workers, All Other"
616,Liaison Staff (Construction),Payroll and Timekeeping Clerks
1050,Store Keeper,General and Operations Managers
750,Agricultural RPA Pilot,Commercial Pilots
217,Ground Operations Crew Member,Flight Attendants
937,Instrumentation And Automation Technician,Industrial Engineering Technologists and Techn...
43,Community Media Coordinator,Audio and Video Technicians
791,Engineered Bamboo Processor,"Woodworking Machine Setters, Operators, and Te..."
385,Court Clerk,Judicial Law Clerks


In [ ]:
is_instructor = (mca_df['PSOC Code'] == '2310')
mca_df.loc[is_37_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
103,Ethics Research Assistant,25-1192.00,"Family and Consumer Sciences Teachers, Postsec..."
110,Development Researcher,25-1192.00,"Family and Consumer Sciences Teachers, Postsec..."
126,Humanities Researcher,25-1125.00,"History Teachers, Postsecondary"
129,Legislative Researcher,25-1065.00,"Political Science Teachers, Postsecondary"
197,Religious Research Assistant,25-1126.00,"Philosophy and Religion Teachers, Postsecondary"
506,Anatomy and Physiology Instructor,25-1071.00,"Health Specialties Teachers, Postsecondary"
514,Biology Instructor,25-1042.00,"Biological Science Teachers, Postsecondary"
645,Mathematics Instructor,25-1022.00,"Mathematical Science Teachers, Postsecondary"
673,Philosophy and Ethics Instructor,25-1126.00,"Philosophy and Religion Teachers, Postsecondary"


In [ ]:
is_instructor = (mca_df['Job Title'] == 'Mathematics Teacher')
mca_df.loc[is_instructor, ['Job Title', 'SOC Title']]

,Job Title,SOC Title
646,Mathematics Teacher,"Secondary School Teachers, Except Special and ..."


In [ ]:
is_19_SOC_codes = (mca_df['SOC Codes'].apply(len) == 19)
mca_df.loc[is_19_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
8,Urban Forestry Aide,19-1031.03,Park Naturalists
104,Photo Editing Assistant,51-9151.00,Photographic Process Workers and Processing Ma...
151,Electrical Systems Technician,17-3023.00,Electrical and Electronic Engineering Technolo...
181,Maintenance Technician,17-3027.01,Automotive Engineering Technicians
196,HVAC Technician,17-3023.00,Electrical and Electronic Engineering Technolo...
214,Forestry Extension Assistant,25-9021.00,Farm and Home Management Educators
239,IT Support Technician (Air Transport),17-3023.00,Electrical and Electronic Engineering Technolo...
253,Chemical Lab Technologist,17-3025.00,Environmental Engineering Technologists and Te...
268,Electronics Repair Technician,17-3023.00,Electrical and Electronic Engineering Technolo...
272,IT Technician,17-3023.00,Electrical and Electronic Engineering Technolo...


In [ ]:
is_11_SOC_codes = (mca_df['SOC Codes'].apply(len) == 11)
mca_df.loc[is_11_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
176,Operations Support Analyst,15-2031.00,Operations Research Analysts
216,Flight Dispatch Officer,43-5032.00,"Dispatchers, Except Police, Fire, and Ambulance"
279,Population Data Scientist,15-2041.00,Statisticians
305,Vector Control Assistant,53-1043.00,First-Line Supervisors of Material-Moving Mach...
329,Food Safety Auditor,45-2011.00,Agricultural Inspectors
387,Metadata Specialist,15-2099.01,Bioinformatics Technicians
428,Police Officer,33-3051.00,Police and Sheriff's Patrol Officers
450,Sanitation Inspector,53-1042.01,Recycling Coordinators
496,Actuarial Assistant (Healthcare),15-2011.00,Actuaries
499,Advertising Clerk (Tourism),43-4181.00,Reservation and Transportation Ticket Agents a...


In [ ]:
mca_df.loc[[3, 4], ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
3,Financial Auditor,13-2011.00,Accountants and Auditors
4,Tax Accountant,13-2082.00,Tax Preparers
